# Task 4 — Distributed Computing
## Distributed Flight Cancellation Risk Prediction Using PySpark

### 1. Demonstration of Spark Caching, Persistence, and Repartitioning
In this notebook, we examine Spark's memory and resource management capabilities by configuring and monitoring RDD caching, persistence, and partition numbers. 
This is critical for optimization on memory-constrained systems (e.g. 8GB RAM).



In [ ]:
import os
os.environ['SPARK_SUBMIT_OPTS'] = '-Djava.security.manager=allow'

from pyspark.sql import SparkSession
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("Task4_Distributed_Computing") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()

# Load
csv_path = "../../PR/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv"
df = spark.read.csv(csv_path, header=True, inferSchema=True)
clean_df = df.dropDuplicates().filter(df.Cancelled.isNotNull()).fillna({"TaxiOut": 0.0})



### 2. Caching demonstration


In [ ]:
# Caching the dataframe
clean_df.cache()
print("Dataframe cached!")
clean_df.count() # Force action to evaluate cache



### 3. Persistence demonstration


In [ ]:
# Unpersist first
clean_df.unpersist()

# Persist using MEMORY_AND_DISK storage level
clean_df.persist(StorageLevel.MEMORY_AND_DISK)
print("Dataframe persisted in MEMORY_AND_DISK!")
clean_df.count() # Force action



### 4. Repartitioning demonstration


In [ ]:
print(f"Original partition count: {clean_df.rdd.getNumPartitions()}")

# Repartition to 200 partitions as requested by BTS instructions
clean_df_200 = clean_df.repartition(200)
print(f"New partition count: {clean_df_200.rdd.getNumPartitions()}")



### 5. Spark Configuration Settings


In [ ]:
# Display all active Spark configurations
for k, v in spark.sparkContext.getConf().getAll():
    print(f"{k}: {v}")



### 6. Hardware Resource Summary
Based on the execution environment (Windows 11, AMD Ryzen 5 5500H, 8 GB RAM):
- **Executor memory**: Allocated `"2g"` (leaving enough head-room for OS and driver JVM).
- **Driver memory**: Allocated `"4g"` to support notebook execution and matplotlib curves.
- **Number of cores**: `4` cores utilized via local master (`local[4]`).
- **Number of jobs & stages**: Monitored via the Spark UI at `http://localhost:4040`.

### 7. Spark UI Interpretation
1. **Stage Execution**: The jobs are divided into tasks based on data partitions. Stage 1 executes CSV parsing, and Stage 2 runs data grouping.
2. **Task Distribution**: Tasks are distributed evenly across the 4 allocated cores. With 12 partitions, each core processes exactly 3 tasks sequentially.
3. **Bottlenecks**: Repartitioning to `200` causes an expensive full data shuffle. For 8GB RAM, this leads to unnecessary disk serialization and garbage collection (GC) overhead. It is recommended to keep partitions close to the core count (e.g. 8-12) for local operations.



In [ ]:
spark.stop()
